# Outlet Spot-Check — Verify Scheme Discount Deduplication

**Purpose:** Pick one outlet and trace it across multiple months to confirm:
1. Qty / Sale Value are taken **once per invoice line** (no double counting)
2. Scheme Discount (₹) is **summed across all scheme rows** for that line (correct — additive)
3. Compare: raw total vs deduplicated total side by side

In [ ]:
import pandas as pd
import os
import re
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent  # scheme_utilisation folder
RAW_FOLDER   = PROJECT_ROOT / "Raw files"

# ---- Configuration ----
TARGET_OUTLET = 91118          # Outlet we are tracing
CHECK_MONTHS  = [              # Filenames of months to check
    "Apr 24 Scheme Utilization.xlsx",
    "May 24 Scheme Utilization.xlsx",
    "June 24 Scheme Utilization.xlsx",
]

COLS_NEEDED = [
    "SUb_Brand", "Distributor_Type", "Outlet_Id",
    "Distributor code", "Distributor_id", "Bill No",
    "Invoice Date", "skunitid", "skucode", "Batch_Id", "Batch No",
    "Scheme Name", "Scheme_Reason", "%_Scheme",
    "Invoice qty. pieces", "Scheme_discount",
    "Distributor_Sale_Value", "Sale_Value",
]

LINE_KEY_COLS = [
    "Distributor code", "Distributor_id", "Outlet_Id", "Bill No",
    "Invoice Date", "skunitid", "skucode", "Batch_Id", "Batch No",
]

print(f"Checking outlet: {TARGET_OUTLET}")
print(f"Across months  : {[m.split()[0]+' '+m.split()[1] for m in CHECK_MONTHS]}")

In [ ]:
def load_month(filename):
    """Load one raw file, filter to STOCKIEST DMS + 12/18 ML + no SAMT."""
    df = pd.read_excel(RAW_FOLDER / filename, usecols=COLS_NEEDED, dtype={"Outlet_Id": object})

    df = df[df["Distributor_Type"].str.strip() == "STOCKIEST DMS"]
    df = df[df["SUb_Brand"].str.upper().str.contains("12 ML|18 ML", na=False)]
    df = df[~df["Scheme Name"].str.upper().str.contains("SAMT", na=False)]
    return df


# Load selected months and tag with month label
frames = []
for fname in CHECK_MONTHS:
    month_label = " ".join(fname.split()[:2])
    df = load_month(fname)
    df["_month"] = month_label
    frames.append(df)
    print(f"{month_label}: {len(df):,} rows after base filters")

all_data = pd.concat(frames, ignore_index=True)

## Step 1 — Raw rows for our target outlet (before dedup)

In [ ]:
# All raw rows for this outlet across the selected months
outlet_raw = all_data[all_data["Outlet_Id"].astype(str) == str(TARGET_OUTLET)].copy()

print(f"Raw rows for outlet {TARGET_OUTLET}: {len(outlet_raw)}")
print()

display_cols = ["_month", "SUb_Brand", "Bill No", "Invoice Date",
                "Scheme Name", "Scheme_Reason", "%_Scheme",
                "Invoice qty. pieces", "Scheme_discount",
                "Distributor_Sale_Value", "Sale_Value"]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)
outlet_raw[display_cols].sort_values(["_month", "Bill No", "Scheme_Reason"]).reset_index(drop=True)

## Step 2 — Naive sum WITHOUT dedup (shows double counting)

In [ ]:
# If we just summed everything naively — qty and sale values would be doubled
naive = (
    outlet_raw
    .groupby("_month", sort=False)
    .agg(
        raw_row_count        = ("Invoice qty. pieces", "count"),
        naive_qty            = ("Invoice qty. pieces", "sum"),
        naive_scheme_disc    = ("Scheme_discount", "sum"),
        naive_dist_sale_val  = ("Distributor_Sale_Value", "sum"),
        naive_sale_val       = ("Sale_Value", "sum"),
    )
    .reset_index()
    .rename(columns={"_month": "Month"})
)
print("NAIVE (no dedup) — note inflated qty and sale values:")
naive

## Step 3 — Correct dedup: Qty & Sale Value taken ONCE per invoice line, Scheme Discount summed

In [ ]:
outlet_raw["_line_key"] = outlet_raw[LINE_KEY_COLS].astype(str).agg("|".join, axis=1)

# For each invoice line: qty and sale values come from FIRST row only
first_occurrence = outlet_raw.drop_duplicates(subset="_line_key", keep="first")[
    ["_month", "_line_key", "Invoice qty. pieces", "Distributor_Sale_Value", "Sale_Value"]
]

# Scheme discount is additive across all scheme rows for the same line
scheme_disc_per_line = (
    outlet_raw
    .groupby(["_month", "_line_key"])["Scheme_discount"]
    .sum()
    .reset_index()
)

# Merge: one row per invoice line with correct values
deduped = first_occurrence.merge(scheme_disc_per_line, on=["_month", "_line_key"])

print(f"Unique invoice lines for outlet {TARGET_OUTLET}: {len(deduped)}")
print()
deduped.drop(columns="_line_key").sort_values(["_month"]).reset_index(drop=True)

## Step 4 — Month-wise comparison: Naive vs Correct

In [ ]:
correct = (
    deduped
    .groupby("_month", sort=False)
    .agg(
        unique_lines         = ("_line_key" if "_line_key" in deduped.columns else "Invoice qty. pieces", "count"),
        correct_qty          = ("Invoice qty. pieces", "sum"),
        correct_scheme_disc  = ("Scheme_discount", "sum"),
        correct_dist_sale_val= ("Distributor_Sale_Value", "sum"),
        correct_sale_val     = ("Sale_Value", "sum"),
    )
    .reset_index()
    .rename(columns={"_month": "Month"})
)

comparison = naive.merge(correct, on="Month")
comparison["qty_inflation"]      = comparison["naive_qty"] / comparison["correct_qty"]
comparison["sale_val_inflation"] = comparison["naive_sale_val"] / comparison["correct_sale_val"]

print(f"=== Outlet {TARGET_OUTLET} — Naive vs Correct ===")
print()
print(comparison[[
    "Month",
    "raw_row_count", "unique_lines",
    "naive_qty", "correct_qty", "qty_inflation",
    "naive_scheme_disc", "correct_scheme_disc",
    "naive_sale_val", "correct_sale_val", "sale_val_inflation",
]].to_string(index=False))

## Interpretation

- **`qty_inflation` = 2.0** → means naive approach double-counted quantity (2 scheme rows per invoice line)
- **`qty_inflation` = 1.0** → no issue for that month
- **`naive_scheme_disc` = `correct_scheme_disc`** → scheme discount should always match (it IS additive)
- The main notebook correctly uses deduplication so the `correct_*` numbers are what end up in the output